# Consumo ilimitado: o link que vazou num fórum

OWASP Top 10 para LLMs 2025 — **LLM10 Unbounded Consumption**

Simulador didático (`campus/`), sem rede e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [1]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, narrar  # noqa: E402
from campus.llm import SimLLM  # noqa: E402

print("Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.")

Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.


## Cenário
Terça-feira: cerca de 400 chamadas de alunos ao Aurora Assist. Quinta-feira, um fórum externo compartilha o endpoint e poucos endereços passam a usá-lo como tradutor gratuito: cerca de 38 mil chamadas no dia. Custo simulado: US$ 0,029 por chamada.

In [2]:
CUSTO_POR_CHAMADA = 0.029  # dólares, valor simulado
LIMITE_POR_ORIGEM = 50      # chamadas por origem por dia

def dia(terceiros):
    """200 alunos (2 chamadas cada) + `terceiros` chamadas vindas de 5 endereços externos."""
    alunos = ["aluno-{:03d}".format(i % 200) for i in range(400)]
    externos = ["externo-{}".format(i % 5) for i in range(terceiros)]
    return alunos + externos

def simular(chamadas, limitador=None):
    atendidas = [o for o in chamadas if limitador is None or limitador.permitir(o)]
    alunos_bloqueados = sum(1 for o in chamadas if o.startswith("aluno")) - \
        sum(1 for o in atendidas if o.startswith("aluno"))
    return len(atendidas) * CUSTO_POR_CHAMADA, len(chamadas) - len(atendidas), alunos_bloqueados

terca, quinta = dia(0), dia(37600)

## Sem defesa

In [3]:
for nome, chamadas in (("terça", terca), ("quinta", quinta)):
    custo, _, _ = simular(chamadas)
    print("{:<7} {:>6} chamadas  US$ {:>8.2f}".format(nome, len(chamadas), custo))
assert simular(quinta)[0] > 1000

terça      400 chamadas  US$    11.60
quinta   38000 chamadas  US$  1102.00


## Com defesa
`defesas.LimitadorTaxa` limita as chamadas **por origem**. Cada aluno é uma origem própria, então o uso legítimo não é afetado.

In [4]:
custo, bloqueadas, alunos_bloqueados = simular(quinta, defesas.LimitadorTaxa(LIMITE_POR_ORIGEM))
print("quinta com limite: US$ {:.2f}  ({} chamadas bloqueadas, {} delas de alunos)".format(
    custo, bloqueadas, alunos_bloqueados))
assert custo < 20 and alunos_bloqueados == 0

quinta com limite: US$ 18.85  (37350 chamadas bloqueadas, 0 delas de alunos)


## O que observar
- Nenhuma chamada isolada parecia maliciosa: o problema é o volume vindo de poucas origens.
- "Denial of wallet": o serviço continua no ar e a conta explode; sem alerta, só se percebe na fatura.
- O limite precisa ser por usuário/origem. Um limite único para "alunos" bloquearia o uso legítimo junto com o abuso.
- Limite de taxa é o controle mais barato de ligar no dia 1; quota por usuário, teto de tokens, autenticação do endpoint e alerta de custo completam a defesa.